In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
     from dlroms import*
except:
     !pip install --no-deps git+https://github.com/NicolaRFranco/dlroms.git
     from dlroms import*

# **Lab 3 - Reduced Basis method (Part 1: stationary PDEs)**

The Reduced Basis (RB) method is an intrusive model order reduction technique for parametrized PDEs. Consider a stationary **linear** PDE depending on a parameter $\boldsymbol{\mu}\in\mathbb{R}^{p}$. Discretizing the problem via the Finite Element Method (FEM) yields a linear system of the form

$$\begin{equation}\tag{1}\mathbf{A}_{\boldsymbol{\mu}}\mathbf{u}=\mathbf{f}_{\boldsymbol{\mu}},\end{equation}$$

where $\mathbf{A}_{\boldsymbol{\mu}}\in\mathbb{R}^{N_{h}\times N_{h}}$ and $\mathbf{f}_{\boldsymbol{\mu}}\in\mathbb{R}^{N_{h}}$ are parameter dependent. Here, $N_{h}$ is the dimension of the underlying Finite Element space $V_{h}$.
</br></br>
**Eq. (1) is referred to as the Full Order Model (FOM)**. A FOM solver is a suitable algorithm that, given $\boldsymbol{\mu}$ computes the solution $\mathbf{u}=\mathbf{u}_\boldsymbol{\mu}$ of (1).
</br>
</br>
The idea behind model order reduction (or "reduced order modeling"), is that, once we have solved (1) several times, we may note that PDE solutions tend to exhbit recurring patterns, which may allow us to take suitable short-cuts when computing new solutions. Specifically, in the RB approach, we ask ourselves the following question:
</br></br>

<p align="center"><i>Is it worth solving the PDE in the whole FOM space?
</br>Maybe there is a better basis, specifically tailored for this parametrized problem...!</i></p></br>

**In this lab, we shall explore this concept by re-visiting the parametrized cookie problem (see Lab. 2, Ex. 5).**

In [ ]:
from dlroms.testcases.cookies import Vh, FOMsolver

mu0 = np.array([1, 0.1, 1, 1])
u0 = FOMsolver(mu0)

plt.figure(figsize = (4, 4))
fe.plot(u0, Vh, colorbar = True)
plt.title("Example of a FOM solution")
plt.show()

In [ ]:
# Data generation [...]







In [ ]:
# Extra: we can also store FOM simulations and load them later.
# To this end, there are at least two ways
#
# 1) Use np.save and np.load to save mu and u separately, namely
#
#    np.save("params.npy", mu)
#    np.save("soluts.npy", u)
#
#    These can then be loaded, even in a different notebook, via
#
#    mu = np.load("params.npy")
#    u = np.load("soluts.npy")
#
# 2) Save both mu and u in a single "zipped" file using np.savez
#
#    np.savez("FOMdata.npz", params = mu, soluts = u)
#
#    To load and access the simulations, we can then use
#
#    simulations = np.load("FOMdata.npz")
#    mu = simulations['params']
#    u = simulations['soluts']

# The RB method in a nutshell

The idea is to replace (1) with a smaller system, obtained by substituting the FOM space, $V_{h}$, with a smaller one, $V_{rb}\subset V_{h}$. To see how this works, let $\mathbf{V}\in\mathbb{R}^{N_{h}\times n}$ be an orthogonal matrix, representing a suitable basis of $V_{rb}$, where $\text{dim}(V_{rb})=n$. Let

$$\mathbf{A}_{\boldsymbol{\mu}}^{rb}:= \mathbf{V}^{\top}\mathbf{A}_{\boldsymbol{\mu}}\mathbf{V},\quad\quad \mathbf{f}^{rb}_{\boldsymbol{\mu}}:=\mathbf{V}^{\top}\mathbf{f}_{\boldsymbol{\mu}}.$$

Then, it is straight forward to see that the $n\times n$ system below

$$\begin{equation}\tag{2}\mathbf{A}_{\boldsymbol{\mu}}^{rb}\mathbf{c}^{rb}=\mathbf{f}^{rb}_{\boldsymbol{\mu}},\end{equation}$$

is nothing but the projection of (1) onto the reduced space $V_{rb}$. Equivalently, we are considering a Galerkin projection of the PDE onto the RB space, rather than onto the Finite Element space.

Thus, an RB solver works as follows. Given a parameter instance $\boldsymbol{\mu}$,

1. Assemble $\mathbf{A}_{\boldsymbol{\mu}}^{rb}$ and $\mathbf{f}_{\boldsymbol{\mu}}^{rb}$;

2. Solve (2), retriving $\mathbf{c}^{rb}=\mathbf{c}^{rb}_{\boldsymbol{\mu}}\in\mathbb{R}^{n}$;

3. Convert the RB solution, from RB notation to FOM notation (change of basis):

$$\mathbf{u}^{rb}_{\boldsymbol{\mu}} := \mathbf{V}\mathbf{c}^{rb}_{\boldsymbol{\mu}}.$$

Operatively, an RB solver, must address the following questions:

Q1: how do we choose a suitable RB space?

Q2: how do we assemble the reduced system efficiently?

## 1. Proper Orthogonal Decomposition (POD)

In order to pick a suitable basis, we look for possible matrices $\mathbf{V}$ for which
$$\mathbf{u}_{\boldsymbol{\mu}}\approx \mathbf{V}\mathbf{V}^{\top}\mathbf{u}_{\boldsymbol{\mu}}.$$

In practice, this can be achieved by collecting a random sample of PDE solutions and performing a suitable truncated Singular Value Decomposition (SVD). Let $\mathbf{u}_{i}:=\mathbf{u}_{\boldsymbol{\mu}_{i}}$. We run the FOM solver $N$ times to collect $N$ different simulations
$[\mathbf{u}_{1},\dots, \mathbf{u}_{N}]$,
each one associated to a random parameter instance $\boldsymbol{\mu}_{i}$.

Then, we can collect all the simulations in a matrix (called **snapshots matrix**)
$$\mathbf{U}_{\text{train}}=[\mathbf{u}_{1},\dots, \mathbf{u}_{N}]\in\mathbb{R}^{N_{h}\times N}$$

of which we perform an SVD decomposition,

$$\mathbf{X}\mathbf{S}\mathbf{Y}^{\top}=\mathbf{U}_\text{train}.$$

Here, $\mathbf{S}=\text{diag}(s_{1},\dots, s_{N})$ are the singular values of $\mathbf{U}$, listed in decreasing order.

The idea is to define $\mathbf{V}$ by extracting the first $n$ columns of $\mathbf{X}$. This procedure is called **Proper Orthogonal Decomposition**.


<mark>**Exercise 1**</mark></br>
Split the 100 simulations into a training set of size $N=50$ and a test set of size $N_{\text{test}}=50$.

 Leveraging on the function $\texttt{svd}$ in the $\texttt{scipy.linalg}$ package, compute the SVD of the snapshots matrix $\mathbf{U}_\text{train}$.

1. Plot the singular values $s_{1},s_{2},\dots,s_{N}$. Given the decay of the singular values, pick a suitable reduced dimension $n$. NB: what happens if you exclude $s_{1}$ when doing the plot?

2. Construct the POD matrix $\mathbf{V}$ and plot the first three basis functions.

3. What is the average projection error entailed by the POD space? Compute it by evaluating the formula below</br></br>
$$E_{\text{proj}}=\frac{1}{N_{\text{test}}}\sum_{i=1}^{N_\text{test}}\frac{|\mathbf{u}_{i}^{\text{test}}-\mathbf{V}\mathbf{V}^{\top}\mathbf{u}_{i}^{\text{test}}|}{|\mathbf{u}_{i}^{\text{test}}|}$$</br>where $\mathbf{u}_{i}^{\text{test}}$ are the simulations in the test set.


In [ ]:
from scipy.linalg import svd





In [ ]:
# 1. Plotting the singular values






In [ ]:
# 2. POD matrix






In [ ]:
# 3. Projection error
from dlroms import num2p






## 2. Assembling the ROM (affine case)

Even with the POD matrix at hand, assembling the ROM can be highly nontrivial. Luckily, things become fairly simple in the case of affinely-parametrized operators. In fact, assume that

$$\mathbf{A}_{\boldsymbol{\mu}}=\mathbf{A}_{0}+\sum_{j=1}^{p}\mu_{j}\mathbf{A}_{j}$$

for some fixed $\mathbf{A}_{0},\dots,\mathbf{A}_{p}$. For simplicity, say that $\mathbf{f}_{\boldsymbol{\mu}}=\mathbf{f}$ is constant in $\boldsymbol{\mu}$. Then, we can pre-compute

- $\mathbf{f}_{\boldsymbol{\mu}}^{rb}:=\mathbf{V}^{\top}\mathbf{f}$;

- $\mathbf{A}_{j}^{rb}:=\mathbf{V}^{\top}\mathbf{A}_{j}\mathbf{V}$ for $j=0,\dots,p$.

Then, during the "online phase", given a parameter instance $\boldsymbol{\mu}$, assembling $\mathbf{A}_{\boldsymbol{\mu}}^{rb}$ boils down to computing
</br>
$$\mathbf{A}_{\boldsymbol{\mu}}^{rb}=\mathbf{A}_{0}^{rb}+\sum_{j=1}^{p}\mu_{j}\mathbf{A}_{j}^{rb}.$$

<mark>**Exercise 2**</mark></br>
Pre-compute $\mathbf{f}_{\boldsymbol{\mu}}^{rb}$ and $\mathbf{A}_{j}^{rb}$ for $j=0,\dots,p$. Then, construct a function called $\texttt{assembleROM}$ that, given $\boldsymbol{\mu}\in\mathbb{R}^{p}$ returns $\mathbf{A}_{\boldsymbol{\mu}}^{rb}$ and $\mathbf{f}_{\boldsymbol{\mu}}^{rb}$.


In [ ]:
from dlroms.testcases.cookies import A_out, A0, A1, A2, A3, fh

# TO DO
# TO DO
# TO DO

In [ ]:
def assembleROM(mu):

  # TO DO
  # TO DO
  # TO DO

  return Arb, frb

## 3. Let's try this out!

<mark>**Exercise 3**</mark></br>
Construct a function called $\texttt{RBsolver}$ that, given $\boldsymbol{\mu}\in\mathbb{R}^{p}$ returns the RB solution $\mathbf{u}_{\boldsymbol{\mu}}^{rb}$ in FOM coordinates.

1. Let $\boldsymbol{\mu}_{0}:=[0.2, 0.01, 1, 0.4]$. Compare FOM and ROM solutions (plot, relative error, etc.).

2. How fast is the ROM compared to the FOM?

3. What is the average relative error of the ROM across the whole test set?


In [ ]:
def RBsolver(mu):

  # TO DO
  # TO DO
  # TO DO

  return urb

In [ ]:
# 1. Comparison for a specific mu







In [ ]:
# 2. Time complexity
from time import perf_counter






In [ ]:
#3. Test error




